In [3]:
import os
import sys
sys.path.append("../") # go to parent dir
%load_ext autoreload
%autoreload 2

In [4]:
from ultralytics import YOLO


# Load the train and test datasets

In [5]:
from functions.loading_functions import *
train_df = get_dataset("train")
test_df = get_dataset("test")

# Loading the pretrained model found here(https://huggingface.co/foduucom/plant-leaf-detection-and-classification)

In [ ]:
from ultralytics import YOLO

model = YOLO("plant-leaf-detection-and-classification/best.pt")

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil
import yaml
import torch

# Step 1: Process the original dataset and convert to YOLO format
def process_dataset(df, output_dir='yolo_dataset'):
    """
    Convert the original dataset format to YOLO format.
    
    Args:
        df: pandas DataFrame with columns ['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']
        output_dir: directory to save the YOLO format dataset
    """
    # Create necessary directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    
    # Get unique classes and create class mapping
    classes = df['class'].unique().tolist()
    class_to_id = {class_name: i for i, class_name in enumerate(classes)}
    
    # Save the class mapping to a file
    with open(os.path.join(output_dir, 'classes.txt'), 'w') as f:
        for class_name in classes:
            f.write(f"{class_name}\n")
    
    # Process each image
    processed_images = set()
    for _, row in df.iterrows():
        img_path = row['ImagePath']
        img_id = row['Image_ID']
        
        # Skip if we've already processed this image
        if img_id in processed_images:
            continue
        
        # Copy the image to the YOLO dataset
        dst_img_path = os.path.join(output_dir, 'images', img_id)
        shutil.copy(img_path, dst_img_path)
        
        # Create a label file for this image
        label_filename = img_id.split(".")[0]+".txt"
        label_path = os.path.join(output_dir, 'labels', label_filename)
        
        # Get all annotations for this image
        img_annotations = df[df['Image_ID'] == img_id].copy()
        
        # Open the image to get dimensions
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # Write annotations in YOLO format
        with open(label_path, 'w') as f:
            for _, ann in img_annotations.iterrows():
                # Convert bbox coordinates to YOLO format (normalized center x, center y, width, height)
                x_min, y_min = ann['xmin'], ann['ymin']
                x_max, y_max = ann['xmax'], ann['ymax']
                
                # Normalize coordinates
                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height
                
                # Get class ID
                class_id = class_to_id[ann['class']]
                
                # In case they are none
                # Write in YOLO format: class_id x_center y_center width height
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
        
        processed_images.add(img_id)
    
    print(f"Processed {len(processed_images)} images with {len(df)} annotations.")
    return classes

def split_dataset(output_dir='yolo_dataset', train_ratio=0.8, val_ratio=0.2):
    """
    Split the dataset into train and validation sets (no test set).

    Args:
        output_dir: directory containing the YOLO format dataset
        train_ratio: ratio of training data
        val_ratio: ratio of validation data
    """
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

    # Create train and val directories
    for split in ['train', 'val']:
        for subdir in ['images', 'labels']:
            os.makedirs(os.path.join(output_dir, split, subdir), exist_ok=True)

    # Split the dataset into train and validation
    train_images, val_images = train_test_split(all_images, train_size=train_ratio, random_state=42)

    # Move files to respective directories
    for split, images in [('train', train_images), ('val', val_images)]:
        for img_file in images:
            # Get corresponding label file
            label_file = os.path.splitext(img_file)[0] + '.txt'

            # Move image
            src_img = os.path.join(output_dir, 'images', img_file)
            dst_img = os.path.join(output_dir, split, 'images', img_file)
            shutil.copy(src_img, dst_img)

            # Move label
            src_label = os.path.join(output_dir, 'labels', label_file)
            dst_label = os.path.join(output_dir, split, 'labels', label_file)
            if os.path.exists(src_label):  # Some images might not have annotations
                shutil.copy(src_label, dst_label)

    print(f"Dataset split: {len(train_images)} train, {len(val_images)} validation images.")

def create_yaml_config(classes, output_dir='yolo_dataset'):
    """
    Create a YAML configuration file for YOLOv5.
    
    Args:
        classes: list of class names
        output_dir: directory containing the YOLO format dataset
    """
    config = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(classes),
        'names': classes
    }
    
    with open(os.path.join(output_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"Created YAML configuration file at {os.path.join(output_dir, 'dataset.yaml')}")

def train_yolov11(dataset_yaml, model_name_path: str, model_size='s', epochs=100, batch_size=16, image_size=640):
    """
    Train a YOLOv11 model with focus on mAP50 metric.
    
    Args:
        dataset_yaml: path to the YAML configuration file
        model_name_path: path to the base model
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: number of training epochs
        batch_size: batch size
        image_size: input image size
    """
    # Install Ultralytics package (which includes YOLOv11)
    os.system('pip install ultralytics')
    
    import ultralytics
    from ultralytics import YOLO
    
    # Print Ultralytics version for reference
    print(f"Ultralytics version: {ultralytics.__version__}")
    
    # Load the base model
    model = YOLO(model_name_path)
    
    # Train the model using the dataset YAML config
    # Set metrics to focus on mAP50
    results = model.train(
        data=dataset_yaml,
        epochs=epochs,
        batch=batch_size,
        device=[0],  # List all GPU indices to use
        imgsz=image_size,
        patience=50,  # Early stopping patience
        cache=True,
        project='yolov11_plant_detection',
        name=f'yolov11{model_size}_map50_focused',
        save=True,    # Save best model
        pretrained=True,
        verbose=True,
    )
    
    # Return the path to the best model based on mAP50
    best_model_path = results.best
    print(f"Best model (based on mAP50) saved to: {best_model_path}")
    return best_model_path


def run_inference(model_path, test_image_dir, output_dir='predictions', conf_threshold=0.25, iou_threshold=0.45):
    """
    Run inference on test images using YOLOv11 with focus on mAP50.
    
    Args:
        model_path: path to the trained YOLOv11 model
        test_image_dir: directory containing test images
        output_dir: directory to save prediction results
        conf_threshold: confidence threshold for detections
        iou_threshold: IoU threshold for NMS (affects mAP50 calculation)
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model using Ultralytics YOLO
    from ultralytics import YOLO
    model = YOLO(model_path)
    
    # Run inference with specified IoU threshold
    results = model.predict(
        source=test_image_dir,
        conf=conf_threshold,
        iou=iou_threshold,  # Set IoU threshold for NMS
        save=True,
        save_txt=True,
        save_conf=True,
        project=output_dir,
        name='detect_map50',
        verbose=True
    )
    
    # Export results to a CSV file
    predictions = []
    
    for result in results:
        img_name = os.path.basename(result.path)
        boxes = result.boxes
        
        if len(boxes) > 0:
            # Extract coordinates, confidence, and class
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()  # xyxy format (top-left, bottom-right)
                conf = box.conf.item()
                cls = int(box.cls.item())
                class_name = model.names[cls]
                
                predictions.append({
                    'image_name': img_name,
                    'class': class_name,
                    'confidence': conf,
                    'xmin': x1,
                    'ymin': y1,
                    'xmax': x2,
                    'ymax': y2
                })
    
    # Create DataFrame and save
    if predictions:
        pred_df = pd.DataFrame(predictions)
        pred_df.to_csv(os.path.join(output_dir, 'predictions.csv'), index=False)
    
    print(f"Inference complete. Results saved to {output_dir}")


def evaluate_model_map50(model_path, val_data_path, conf_threshold=0.25, iou_threshold=0.5):
    """
    Evaluate model specifically for mAP50 on validation data.
    
    Args:
        model_path: path to the trained YOLOv11 model
        val_data_path: path to validation data (folder or YAML)
        conf_threshold: confidence threshold for detections
        iou_threshold: IoU threshold for evaluation (set to 0.5 for mAP50)
    """
    from ultralytics import YOLO
    model = YOLO(model_path)
    
    # Run validation with focus on mAP50
    metrics = model.val(
        data=val_data_path,
        conf=conf_threshold,
        iou=iou_threshold,  # Set to 0.5 for mAP50
        verbose=True
    )
    
    # Extract mAP50 value
    map50 = metrics.box.map50
    print(f"\nModel mAP50: {map50:.4f}\n")
    
    # Return detailed metrics
    return metrics

# Step 4: Aggregate image-level predictions for multi-label evaluation
def aggregate_predictions(predictions_csv, output_csv='image_level_predictions.csv'):
    """
    Aggregate bounding box predictions to image-level class predictions.
    
    Args:
        predictions_csv: path to CSV file with bounding box predictions
        output_csv: path to save image-level predictions
    """
    # Load predictions
    pred_df = pd.read_csv(predictions_csv)
    
    # Group by image and aggregate classes
    image_level = pred_df.groupby('image_name')['class'].apply(lambda x: list(set(x))).reset_index()
    image_level.rename(columns={'class': 'predicted_classes'}, inplace=True)
    
    # Save to CSV
    image_level.to_csv(output_csv, index=False)
    
    print(f"Aggregated predictions saved to {output_csv}")

In [ ]:
import os
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from collections import Counter
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil
import yaml
import torch

# Step 1: Process the original dataset and convert to YOLO format
def process_dataset(df, output_dir='yolo_dataset'):
    """
    Convert the original dataset format to YOLO format.
    
    Args:
        df: pandas DataFrame with columns ['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax', 'class_id', 'ImagePath']
        output_dir: directory to save the YOLO format dataset
    """
    # Create necessary directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    
    # Get unique classes and create class mapping
    classes = df['class'].unique().tolist()
    class_to_id = {class_name: i for i, class_name in enumerate(classes)}
    
    # Save the class mapping to a file
    with open(os.path.join(output_dir, 'classes.txt'), 'w') as f:
        for class_name in classes:
            f.write(f"{class_name}\n")
    
    # Process each image
    processed_images = set()
    for _, row in df.iterrows():
        img_path = row['ImagePath']
        img_id = row['Image_ID']
        
        # Skip if we've already processed this image
        if img_id in processed_images:
            continue
        
        # Copy the image to the YOLO dataset
        dst_img_path = os.path.join(output_dir, 'images', img_id)
        shutil.copy(img_path, dst_img_path)
        
        # Create a label file for this image
        label_filename = img_id.split(".")[0]+".txt"
        label_path = os.path.join(output_dir, 'labels', label_filename)
        
        # Get all annotations for this image
        img_annotations = df[df['Image_ID'] == img_id].copy()
        
        # Open the image to get dimensions
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # Write annotations in YOLO format
        with open(label_path, 'w') as f:
            for _, ann in img_annotations.iterrows():
                # Convert bbox coordinates to YOLO format (normalized center x, center y, width, height)
                x_min, y_min = ann['xmin'], ann['ymin']
                x_max, y_max = ann['xmax'], ann['ymax']
                
                # Normalize coordinates
                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width = (x_max - x_min) / img_width
                height = (y_max - y_min) / img_height
                
                # Get class ID
                class_id = class_to_id[ann['class']]
                
                # In case they are none
                # Write in YOLO format: class_id x_center y_center width height
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
        
        processed_images.add(img_id)
    
    print(f"Processed {len(processed_images)} images with {len(df)} annotations.")
    return classes

def split_dataset(output_dir='yolo_dataset', train_ratio=0.8, val_ratio=0.2):
    """
    Split the dataset into train and validation sets (no test set).

    Args:
        output_dir: directory containing the YOLO format dataset
        train_ratio: ratio of training data
        val_ratio: ratio of validation data
    """
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

    # Create train and val directories
    for split in ['train', 'val']:
        for subdir in ['images', 'labels']:
            os.makedirs(os.path.join(output_dir, split, subdir), exist_ok=True)

    # Split the dataset into train and validation
    train_images, val_images = train_test_split(all_images, train_size=train_ratio, random_state=42)

    # Move files to respective directories
    for split, images in [('train', train_images), ('val', val_images)]:
        for img_file in images:
            # Get corresponding label file
            label_file = os.path.splitext(img_file)[0] + '.txt'

            # Move image
            src_img = os.path.join(output_dir, 'images', img_file)
            dst_img = os.path.join(output_dir, split, 'images', img_file)
            shutil.copy(src_img, dst_img)

            # Move label
            src_label = os.path.join(output_dir, 'labels', label_file)
            dst_label = os.path.join(output_dir, split, 'labels', label_file)
            if os.path.exists(src_label):  # Some images might not have annotations
                shutil.copy(src_label, dst_label)

    print(f"Dataset split: {len(train_images)} train, {len(val_images)} validation images.")

def create_yaml_config(classes, output_dir='yolo_dataset'):
    """
    Create a YAML configuration file for YOLOv5.
    
    Args:
        classes: list of class names
        output_dir: directory containing the YOLO format dataset
    """
    config = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(classes),
        'names': classes
    }
    
    with open(os.path.join(output_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"Created YAML configuration file at {os.path.join(output_dir, 'dataset.yaml')}")


def run_inference(model_path, test_image_dir, output_dir='predictions', conf_threshold=0.25, iou_threshold=0.45):
    """
    Run inference on test images using YOLOv11 with focus on mAP50.
    
    Args:
        model_path: path to the trained YOLOv11 model
        test_image_dir: directory containing test images
        output_dir: directory to save prediction results
        conf_threshold: confidence threshold for detections
        iou_threshold: IoU threshold for NMS (affects mAP50 calculation)
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model using Ultralytics YOLO
    from ultralytics import YOLO
    model = YOLO(model_path)
    
    # Run inference with specified IoU threshold
    results = model.predict(
        source=test_image_dir,
        conf=conf_threshold,
        iou=iou_threshold,  # Set IoU threshold for NMS
        save=True,
        save_txt=True,
        save_conf=True,
        project=output_dir,
        name='detect_map50',
        verbose=True
    )
    
    # Export results to a CSV file
    predictions = []
    
    for result in results:
        img_name = os.path.basename(result.path)
        boxes = result.boxes
        
        if len(boxes) > 0:
            # Extract coordinates, confidence, and class
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()  # xyxy format (top-left, bottom-right)
                conf = box.conf.item()
                cls = int(box.cls.item())
                class_name = model.names[cls]
                
                predictions.append({
                    'image_name': img_name,
                    'class': class_name,
                    'confidence': conf,
                    'xmin': x1,
                    'ymin': y1,
                    'xmax': x2,
                    'ymax': y2
                })
    
    # Create DataFrame and save
    if predictions:
        pred_df = pd.DataFrame(predictions)
        pred_df.to_csv(os.path.join(output_dir, 'predictions.csv'), index=False)
    
    print(f"Inference complete. Results saved to {output_dir}")

# Step 4: Aggregate image-level predictions for multi-label evaluation
def aggregate_predictions(predictions_csv, output_csv='image_level_predictions.csv'):
    """
    Aggregate bounding box predictions to image-level class predictions.
    
    Args:
        predictions_csv: path to CSV file with bounding box predictions
        output_csv: path to save image-level predictions
    """
    # Load predictions
    pred_df = pd.read_csv(predictions_csv)
    
    # Group by image and aggregate classes
    image_level = pred_df.groupby('image_name')['class'].apply(lambda x: list(set(x))).reset_index()
    image_level.rename(columns={'class': 'predicted_classes'}, inplace=True)
    
    # Save to CSV
    image_level.to_csv(output_csv, index=False)
    
    print(f"Aggregated predictions saved to {output_csv}")

def calculate_class_weights(dataset_yaml, method='inverse'):
    """
    Calculate class weights based on the distribution of classes in the dataset.
    Useful for analysis, but weights will need manual implementation in YOLO.
    
    Args:
        dataset_yaml: Path to the YAML dataset configuration file
        method: Method to calculate weights ('inverse' or 'effective')
    
    Returns:
        Dictionary mapping class indices to weights
    """
    # Load dataset configuration
    with open(dataset_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Get paths
    dataset_path = Path(data_config['path'])
    train_labels_path = dataset_path / 'train' / 'labels'
    
    # Count occurrences of each class
    class_counts = Counter()
    
    # Process all label files
    for label_file in train_labels_path.glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f:
                if line.strip():
                    class_id = int(line.strip().split()[0])
                    class_counts[class_id] += 1
    
    # Get total number of instances and classes
    total_instances = sum(class_counts.values())
    num_classes = len(data_config['names'])
    
    # Ensure all classes have a count (even if zero)
    for class_id in range(num_classes):
        if class_id not in class_counts:
            class_counts[class_id] = 0
    
    # Calculate weights based on method
    class_weights = {}
    
    if method == 'inverse':
        # Inverse frequency weighting
        for class_id in range(num_classes):
            count = class_counts[class_id]
            if count > 0:
                class_weights[class_id] = total_instances / (num_classes * count)
            else:
                class_weights[class_id] = 1.0  # Default weight for classes with no instances
    
    elif method == 'effective':
        # Effective number of samples weighting
        beta = 0.9999
        for class_id in range(num_classes):
            count = class_counts[class_id]
            if count > 0:
                # Formula: (1 - beta) / (1 - beta^n)
                effective_num = (1.0 - np.power(beta, count)) / (1.0 - beta)
                class_weights[class_id] = 1.0 / effective_num
            else:
                class_weights[class_id] = 1.0
    
    # Normalize weights to have mean of 1
    weight_sum = sum(class_weights.values())
    for class_id in class_weights:
        class_weights[class_id] = class_weights[class_id] * num_classes / weight_sum
    
    print(f"Calculated class weights: {class_weights}")
    
    # Also print the class names with their weights
    class_names = data_config['names']
    print("\nClass weights by name:")
    for class_id, weight in class_weights.items():
        if class_id < len(class_names):
            print(f"  {class_names[class_id]}: {weight:.4f}")
    
    return class_weights

def create_weighted_dataset(dataset_yaml, output_yaml=None, class_weight_method='inverse'):
    """
    Create a weighted dataset by duplicating underrepresented classes in the dataset.
    
    Args:
        dataset_yaml: Path to the original YAML dataset configuration
        output_yaml: Path to save the new YAML configuration (default: adds "_weighted" to original name)
        class_weight_method: Method to calculate class weights ('inverse' or 'effective')
    
    Returns:
        Path to the new weighted dataset YAML file
    """
    # Calculate class weights
    class_weights = calculate_class_weights(dataset_yaml, method=class_weight_method)
    
    # If no output path specified, create one
    if output_yaml is None:
        output_yaml = dataset_yaml.replace('.yaml', '_weighted.yaml')
    
    # Load the original dataset config
    with open(dataset_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Save the class weights for reference
    data_config['class_weights'] = {str(k): float(v) for k, v in class_weights.items()}
    
    # Write the updated config
    with open(output_yaml, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print(f"Created weighted dataset configuration at {output_yaml}")
    print("Note: The class weights are saved for reference only.")
    print("You'll need to use a custom YOLOv11 loss implementation or adjust the cls parameter.")
    
    return output_yaml

def train_yolov11(dataset_yaml, model_name_path, model_size='s', epochs=100, 
                 batch_size=16, image_size=640, cls_gain=0.5, patience=50, 
                 warmup_epochs=3.0, initial_lr=0.01):
    """
    Train a YOLOv11 model with focus on mAP50 metric.
    
    Args:
        dataset_yaml: Path to the YAML configuration file
        model_name_path: Path to the base model
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: Number of training epochs
        batch_size: Batch size
        image_size: Input image size
        cls_gain: Classification loss weight
        patience: Early stopping patience
        warmup_epochs: Number of warmup epochs
        initial_lr: Initial learning rate
    """
    # Install Ultralytics package (which includes YOLOv11)
    os.system('pip install ultralytics')
    
    import ultralytics
    from ultralytics import YOLO
    
    # Print Ultralytics version for reference
    print(f"Ultralytics version: {ultralytics.__version__}")
    
    # Load the base model
    model = YOLO(model_name_path)
    
    # Calculate class weights for analysis (even though we can't directly use them)
    _ = calculate_class_weights(dataset_yaml)
    print("\nNote: Class weights are displayed for reference only.")
    print("YOLOv11 doesn't directly support per-class weights in the training API.")
    print("Consider these alternatives:")
    print("1. Adjust your dataset by oversampling minority classes")
    print("2. Use a higher cls value (e.g., 1.0) to emphasize classification accuracy")
    print("3. Modify the YOLO source code to support class weights\n")
    
    # Train the model with standard parameters
    results = model.train(
        data=dataset_yaml,
        epochs=epochs,
        batch=batch_size,
        device=[0],  # List all GPU indices to use
        imgsz=image_size,
        patience=patience,  # Early stopping patience
        cache=True,
        project='yolov11_plant_detection',
        name=f'yolov11{model_size}_map50_focused',
        save=True,    # Save best model
        pretrained=True,
        verbose=True,
        val=True,     # Enable validation
        save_period=10,  # Save checkpoints every 10 epochs
        # Learning rate settings
        lr0=initial_lr,  # Initial learning rate
        lrf=0.01,  # Final learning rate as a fraction of initial rate
        warmup_epochs=warmup_epochs,
        # Loss function weights - using a potentially higher cls value
        box=7.5,     # Box loss gain
        cls=cls_gain,  # Single value for classification loss
        # Other relevant parameters
        cos_lr=True,  # Use cosine learning rate scheduler
        close_mosaic=10,  # Disable mosaic in last 10 epochs
        amp=True,  # Use mixed precision training
        plots=True  # Generate and save plots
    )
    
    print(f"Training completed. Best model saved based on mAP50.")
    return model


def oversample_minority_classes(dataset_yaml, output_dir=None, threshold_factor=0.5):
    """
    Create a new dataset with oversampled minority classes.
    
    Args:
        dataset_yaml: Path to the YAML dataset configuration
        output_dir: Directory to save the new dataset (default: adds "_balanced" to original dir)
        threshold_factor: Classes with frequency below (max_freq * threshold_factor) will be oversampled
    
    Returns:
        Path to the new dataset YAML file
    """
    print("Creating balanced dataset by oversampling minority classes...")
    
    # Load dataset configuration
    with open(dataset_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Get paths
    dataset_path = Path(data_config['path'])
    
    # Create output directory if not specified
    if output_dir is None:
        output_dir = str(dataset_path) + "_balanced"
    output_dir = Path(output_dir)
    
    # Create necessary directories
    for split in ['train', 'val']:
        for subdir in ['images', 'labels']:
            os.makedirs(output_dir / split / subdir, exist_ok=True)
    
    # Copy validation data as is
    import shutil
    val_img_dir = dataset_path / 'val' / 'images'
    val_lbl_dir = dataset_path / 'val' / 'labels'
    
    for img_file in val_img_dir.glob('*'):
        shutil.copy(img_file, output_dir / 'val' / 'images')
    
    for lbl_file in val_lbl_dir.glob('*'):
        shutil.copy(lbl_file, output_dir / 'val' / 'labels')
    
    print("Copied validation data")
    
    # Count occurrences of each class in training set
    class_counts = Counter()
    train_labels_path = dataset_path / 'train' / 'labels'
    
    # Get all training files
    train_files = []
    for label_file in train_labels_path.glob('*.txt'):
        base_name = label_file.stem
        img_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
        
        # Find corresponding image file
        img_file = None
        for ext in img_extensions:
            potential_img = dataset_path / 'train' / 'images' / (base_name + ext)
            if potential_img.exists():
                img_file = potential_img
                break
        
        if img_file:
            # Read labels to get classes
            with open(label_file, 'r') as f:
                lines = f.readlines()
                file_classes = [int(line.strip().split()[0]) for line in lines if line.strip()]
                
                # Count classes
                for cls in file_classes:
                    class_counts[cls] += 1
                
                # Store file info
                train_files.append({
                    'label_file': label_file,
                    'img_file': img_file,
                    'classes': file_classes
                })
    
    # Determine which classes need oversampling
    max_count = max(class_counts.values()) if class_counts else 0
    threshold = max_count * threshold_factor
    
    minority_classes = {cls: count for cls, count in class_counts.items() if count < threshold}
    
    print(f"Class distribution: {dict(class_counts)}")
    print(f"Identified minority classes: {minority_classes}")
    
    # Copy all original files first
    for file_info in train_files:
        shutil.copy(file_info['img_file'], output_dir / 'train' / 'images')
        shutil.copy(file_info['label_file'], output_dir / 'train' / 'labels')
    
    # Oversample files with minority classes
    duplicated = 0
    for file_info in train_files:
        # Check if file contains any minority class
        if any(cls in minority_classes for cls in file_info['classes']):
            # Calculate duplication factor based on class frequency
            # Files with rarer classes get duplicated more times
            rarest_class = min(
                (cls for cls in file_info['classes'] if cls in minority_classes),
                key=lambda c: minority_classes[c],
                default=None
            )
            
            if rarest_class is not None:
                duplication_factor = int(max_count / minority_classes[rarest_class]) - 1
                duplication_factor = min(duplication_factor, 5)  # Cap at 5x duplication
                
                for i in range(duplication_factor):
                    # Create new filenames for duplicates
                    new_base = f"{file_info['label_file'].stem}_dup{i+1}"
                    new_img = output_dir / 'train' / 'images' / f"{new_base}{file_info['img_file'].suffix}"
                    new_lbl = output_dir / 'train' / 'labels' / f"{new_base}.txt"
                    
                    # Copy files with new names
                    shutil.copy(file_info['img_file'], new_img)
                    shutil.copy(file_info['label_file'], new_lbl)
                    duplicated += 1
    
    print(f"Created {duplicated} duplicate files to balance the dataset")
    
    # Create new YAML file
    new_yaml_path = output_dir / "dataset_balanced.yaml"
    new_config = data_config.copy()
    new_config['path'] = str(output_dir)
    
    with open(new_yaml_path, 'w') as f:
        yaml.dump(new_config, f, default_flow_style=False)
    
    print(f"Created balanced dataset configuration at {new_yaml_path}")
    return str(new_yaml_path)

In [17]:
# Process dataset
classes = process_dataset(train_df)

Processed 5529 images with 9792 annotations.


In [18]:
split_dataset()

Dataset split: 4197 train, 1050 validation images.


In [22]:
create_yaml_config(classes)

Created YAML configuration file at yolo_dataset/dataset.yaml


In [34]:
pretrained_yolo_path = "plant-leaf-detection-and-classification/best.pt"
# Train model with class weights
model = train_yolov11(
    dataset_yaml='yolo_dataset/dataset.yaml',
    model_name_path=pretrained_yolo_path,
    model_size='s',
        epochs=100,
        batch_size=16,
        image_size=640,
        cls_gain=1.0,  # Increase classification weight
        patience=50,
        warmup_epochs=3.0,
        initial_lr=0.01
    )

Ultralytics version: 8.3.94
Calculated class weights: {0: 0.7751539894355594, 1: 1.3052705302174301, 2: 0.9195754803470106}

Class weights by name:
  healthy: 0.7752
  anthracnose: 1.3053
  cssvd: 0.9196

Note: Class weights are displayed for reference only.
YOLOv11 doesn't directly support per-class weights in the training API.
Consider these alternatives:
1. Adjust your dataset by oversampling minority classes
2. Use a higher cls value (e.g., 1.0) to emphasize classification accuracy
3. Modify the YOLO source code to support class weights

New https://pypi.org/project/ultralytics/8.3.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: task=detect, mode=train, model=plant-leaf-detection-and-classification/best.pt, data=yolo_dataset/dataset.yaml, epochs=100, time=None, patience=50, batch=16, imgsz=640, save=True, save_period=10, cache=True, device=[0], workers=8, project

train: Scanning /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/labels.cache... 4197 images, 0 backgrounds, 686 corrupt: 100%|██████████| 4197/4197 [00:00<?, ?it/s]

train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/images/ID_AC3jGA.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2288      1.0926]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/images/ID_AIHFIo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3267      1.1366]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/images/ID_AJD939.jpg: corrupt JPEG restored and saved
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/images/ID_AK7dFo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3218      1.2725]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/train/images/ID_AMshRT.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1269      1.

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (3.1GB RAM): 100%|██████████| 3511/3511 [00:18<00:00, 190.54it/s]
val: Scanning /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset/val/labels.cache... 1050 images, 0 backgrounds, 195 corrupt: 100%|██████████| 1050/1050 [00:00<?, ?it/s]

...

                   all        855       1509      0.713      0.641      0.667      0.417

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    100/100      18.2G     0.9908      1.496      1.445          9        640: 100%|██████████| 220/220 [00:21<00:00, 10.07it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 27/27 [00:03<00:00,  8.44it/s]


                   all        855       1509      0.721      0.634      0.668      0.419

100 epochs completed in 0.713 hours.
Optimizer stripped from yolov11_plant_detection/yolov11s_map50_focused/weights/last.pt, 22.5MB
Optimizer stripped from yolov11_plant_detection/yolov11s_map50_focused/weights/best.pt, 22.5MB

Validating yolov11_plant_detection/yolov11s_map50_focused/weights/best.pt...
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
Model summary (fused): 72 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 27/27 [00:03<00:00,  7.10it/s]


                   all        855       1509       0.72      0.635      0.668      0.419
               healthy        246        619       0.68      0.614      0.643      0.393
           anthracnose        205        319      0.685      0.624       0.63      0.379
                 cssvd        405        571      0.794      0.667      0.731      0.485
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to yolov11_plant_detection/yolov11s_map50_focused
Training completed. Best model saved based on mAP50.


In [35]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from pathlib import Path

def plot_yolo_performance(results_dir, save_plots=False, output_dir='performance_plots', focus_metric='mAP50'):
    """
    Generate comprehensive performance plots for YOLOv11 training results,
    with special focus on mAP@0.5 (IoU threshold of 0.5).
    
    Args:
        results_dir: Directory containing YOLO training results
        save_plots: Whether to save plots to disk (False by default)
        output_dir: Directory to save performance plots (only used if save_plots=True)
        focus_metric: Primary metric to highlight in plots ('mAP50' by default)
    """
    # Create output directory if saving plots
    if save_plots:
        os.makedirs(output_dir, exist_ok=True)
    
    # Check if it's Ultralytics format (YOLOv8/YOLOv11) or older YOLOv5 format
    results_csv = os.path.join(results_dir, 'results.csv')
    
    if os.path.exists(results_csv):
        # Newer Ultralytics format
        df = pd.read_csv(results_csv)
        
        # Check if we have precision-recall data
        if all(col in df.columns for col in ['precision', 'recall', 'mAP50', 'mAP50-95']):
            # Create figure with multiple subplots
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('YOLO Training Performance Metrics\nPrimary Evaluation: mAP@0.5 (IoU=0.5)', fontsize=16)
            
            # Plot Loss curves
            ax = axes[0, 0]
            if 'box_loss' in df.columns and 'cls_loss' in df.columns:
                ax.plot(df['epoch'], df['box_loss'], label='Box Loss')
                ax.plot(df['epoch'], df['cls_loss'], label='Class Loss')
                if 'dfl_loss' in df.columns:  # YOLOv8/v11 specific
                    ax.plot(df['epoch'], df['dfl_loss'], label='DFL Loss')
            else:
                ax.plot(df['epoch'], df['train_loss'], label='Train Loss')
                if 'val_loss' in df.columns:
                    ax.plot(df['epoch'], df['val_loss'], label='Val Loss')
            
            ax.set_title('Training Losses')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot Precision, Recall
            ax = axes[0, 1]
            ax.plot(df['epoch'], df['precision'], label='Precision')
            ax.plot(df['epoch'], df['recall'], label='Recall')
            ax.set_title('Precision and Recall (at IoU=0.5)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Value')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot mAP values with mAP@0.5 highlighted
            ax = axes[1, 0]
            # Plot mAP@0.5 with thicker line and different color
            ax.plot(df['epoch'], df['mAP50'], 'r-', linewidth=3, label='mAP@0.5 (Primary Metric)')
            ax.plot(df['epoch'], df['mAP50-95'], 'b-', linewidth=1.5, label='mAP@0.5:0.95')
            ax.set_title('Mean Average Precision (mAP)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('mAP')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Add a horizontal line at the best mAP@0.5 value
            best_map50 = df['mAP50'].max()
            best_epoch = df.loc[df['mAP50'].idxmax(), 'epoch']
            ax.axhline(y=best_map50, color='r', linestyle='--', alpha=0.5)
            ax.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5)
            ax.annotate(f'Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch:.0f})', 
                        xy=(best_epoch, best_map50),
                        xytext=(best_epoch+2, best_map50-0.02),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
                        fontsize=9)
            
            # Plot learning rate if available, otherwise fitness
            ax = axes[1, 1]
            if 'lr0' in df.columns:
                ax.plot(df['epoch'], df['lr0'], label='Learning Rate')
                ax.set_title('Learning Rate Schedule')
                ax.set_xlabel('Epoch')
                ax.set_ylabel('Learning Rate')
                ax.legend()
                ax.grid(True, linestyle='--', alpha=0.6)
            else:
                # If learning rate is not available, plot fitness if available
                if 'fitness' in df.columns:
                    ax.plot(df['epoch'], df['fitness'], 'g-', label='Fitness')
                    ax.set_title('Model Fitness')
                    ax.set_xlabel('Epoch')
                    ax.set_ylabel('Fitness Value')
                    ax.legend()
                    ax.grid(True, linestyle='--', alpha=0.6)
                else:
                    # Create a special mAP@0.5 focus plot
                    ax.plot(df['epoch'], df['mAP50'], 'ro-')
                    ax.set_title('mAP@0.5 Progress (IoU=0.5)')
                    ax.set_xlabel('Epoch')
                    ax.set_ylabel('mAP@0.5')
                    best_map50 = df['mAP50'].max()
                    ax.axhline(y=best_map50, color='g', linestyle='--', alpha=0.5)
                    ax.text(df['epoch'].max()/2, best_map50*1.01, f'Best: {best_map50:.4f}', 
                            horizontalalignment='center')
                    ax.grid(True, linestyle='--', alpha=0.6)
            
            # Adjust layout
            plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for main title
            
            # Save or display
            if save_plots:
                plt.savefig(os.path.join(output_dir, 'training_metrics.png'), dpi=300)
                plt.close()
            else:
                plt.show()
            
            # Create a dedicated mAP@0.5 (IoU=0.5) plot
            plt.figure(figsize=(10, 6))
            plt.plot(df['epoch'], df['mAP50'], 'ro-', linewidth=2)
            plt.title('mAP@0.5 (IoU=0.5) Training Progress', fontsize=14)
            plt.xlabel('Epoch')
            plt.ylabel('mAP@0.5')
            plt.grid(True, linestyle='--', alpha=0.6)
            
            # Add rolling average to smooth the curve
            window_size = min(5, len(df))
            if window_size > 1:
                rolling_avg = df['mAP50'].rolling(window=window_size).mean()
                plt.plot(df['epoch'][window_size-1:], rolling_avg[window_size-1:], 'b-', 
                         linewidth=1.5, label=f'{window_size}-epoch Moving Average')
            
            # Add best point annotation
            plt.axhline(y=best_map50, color='g', linestyle='--', alpha=0.5)
            plt.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
            plt.annotate(f'Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch:.0f})', 
                        xy=(best_epoch, best_map50),
                        xytext=(best_epoch*(0.9 if best_epoch > df['epoch'].max()/2 else 1.1), 
                                best_map50*(0.95 if best_map50 > df['mAP50'].median() else 1.05)),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                        fontsize=10)
            
            plt.legend()
            plt.tight_layout()
            
            if save_plots:
                plt.savefig(os.path.join(output_dir, 'map50_progress.png'), dpi=300)
                plt.close()
            else:
                plt.show()
            
            # Create PR curve plot if we have confidence data
            pr_curve_file = Path(results_dir) / 'PR_curve.png'
            if pr_curve_file.exists():
                pr_img = plt.imread(pr_curve_file)
                plt.figure(figsize=(10, 8))
                plt.imshow(pr_img)
                plt.axis('off')
                plt.title('Precision-Recall Curve (IoU=0.5)', fontsize=14)
                plt.tight_layout()
                
                if save_plots:
                    plt.savefig(os.path.join(output_dir, 'pr_curve.png'), dpi=300)
                    plt.close()
                else:
                    plt.show()
            
            # Create F1 score vs confidence threshold plot if available
            f1_curve_file = Path(results_dir) / 'F1_curve.png'
            if f1_curve_file.exists():
                f1_img = plt.imread(f1_curve_file)
                plt.figure(figsize=(10, 8))
                plt.imshow(f1_img)
                plt.axis('off')
                plt.title('F1 Score vs Confidence Threshold (IoU=0.5)', fontsize=14)
                plt.tight_layout()
                
                if save_plots:
                    plt.savefig(os.path.join(output_dir, 'f1_curve.png'), dpi=300)
                    plt.close()
                else:
                    plt.show()
            
            # Print final metrics with highlight on mAP@0.5
            final_epoch = df.iloc[-1]
            print("\n" + "="*50)
            print("FINAL TRAINING METRICS")
            print("="*50)
            print(f"Precision (IoU=0.5): {final_epoch['precision']:.4f}")
            print(f"Recall (IoU=0.5): {final_epoch['recall']:.4f}")
            print(f"→ mAP@0.5 (IoU=0.5): {final_epoch['mAP50']:.4f} ←")
            print(f"mAP@0.5:0.95: {final_epoch['mAP50-95']:.4f}")
            print(f"Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch:.0f})")
            print("="*50)
            
            return {
                'precision': final_epoch['precision'],
                'recall': final_epoch['recall'],
                'mAP50': final_epoch['mAP50'],
                'mAP50-95': final_epoch['mAP50-95'],
                'best_mAP50': best_map50,
                'best_epoch': best_epoch
            }
        else:
            print("Warning: CSV file does not contain expected metrics columns.")
    
    # Check for older YOLOv5 format results.txt
    results_txt = os.path.join(results_dir, 'results.txt')
    if os.path.exists(results_txt):
        # Parse results.txt in YOLOv5 format
        epochs, box_loss, obj_loss, cls_loss = [], [], [], []
        precision, recall, map50, map = [], [], [], []
        
        with open(results_txt, 'r') as f:
            for line in f:
                if line.startswith('  Epoch'):
                    continue  # Skip header line
                try:
                    # Extract metrics (format may vary)
                    parts = line.strip().split()
                    if len(parts) >= 12:
                        epoch = int(parts[0])
                        epochs.append(epoch)
                        
                        # Extract losses
                        box_loss.append(float(parts[2]))
                        obj_loss.append(float(parts[3]))
                        cls_loss.append(float(parts[4]))
                        
                        # Extract precision, recall, mAP
                        precision.append(float(parts[8]))
                        recall.append(float(parts[9]))
                        map50.append(float(parts[10]))
                        map.append(float(parts[11]))
                except Exception as e:
                    print(f"Error parsing line: {line}")
                    print(f"Exception: {e}")
        
        if epochs:
            # Create figure with multiple subplots
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('YOLO Training Performance Metrics\nPrimary Evaluation: mAP@0.5 (IoU=0.5)', fontsize=16)
            
            # Plot Loss curves
            ax = axes[0, 0]
            ax.plot(epochs, box_loss, label='Box Loss')
            ax.plot(epochs, obj_loss, label='Objectness Loss')
            ax.plot(epochs, cls_loss, label='Class Loss')
            ax.set_title('Training Losses')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot Precision, Recall
            ax = axes[0, 1]
            ax.plot(epochs, precision, label='Precision')
            ax.plot(epochs, recall, label='Recall')
            ax.set_title('Precision and Recall (at IoU=0.5)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Value')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Plot mAP values with mAP@0.5 highlighted
            ax = axes[1, 0]
            # Plot mAP@0.5 with thicker line and different color
            ax.plot(epochs, map50, 'r-', linewidth=3, label='mAP@0.5 (Primary Metric)')
            ax.plot(epochs, map, 'b-', linewidth=1.5, label='mAP@0.5:0.95')
            ax.set_title('Mean Average Precision (mAP)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('mAP')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Add a horizontal line at the best mAP@0.5 value
            best_map50 = max(map50)
            best_epoch_idx = map50.index(best_map50)
            best_epoch = epochs[best_epoch_idx]
            ax.axhline(y=best_map50, color='r', linestyle='--', alpha=0.5)
            ax.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5)
            ax.annotate(f'Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch})', 
                        xy=(best_epoch, best_map50),
                        xytext=(best_epoch+2, best_map50-0.02),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
                        fontsize=9)
            
            # Plot combined loss
            ax = axes[1, 1]
            total_loss = [b + o + c for b, o, c in zip(box_loss, obj_loss, cls_loss)]
            ax.plot(epochs, total_loss, label='Total Loss')
            ax.set_title('Total Training Loss')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.grid(True, linestyle='--', alpha=0.6)
            
            # Adjust layout
            plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for main title
            
            if save_plots:
                plt.savefig(os.path.join(output_dir, 'training_metrics.png'), dpi=300)
                plt.close()
            else:
                plt.show()
            
            # Create a dedicated mAP@0.5 (IoU=0.5) plot
            plt.figure(figsize=(10, 6))
            plt.plot(epochs, map50, 'ro-', linewidth=2)
            plt.title('mAP@0.5 (IoU=0.5) Training Progress', fontsize=14)
            plt.xlabel('Epoch')
            plt.ylabel('mAP@0.5')
            plt.grid(True, linestyle='--', alpha=0.6)
            
            # Add rolling average to smooth the curve if we have enough data points
            window_size = min(5, len(epochs))
            if window_size > 1:
                # Calculate rolling average manually
                rolling_avg = []
                for i in range(window_size-1, len(map50)):
                    avg = sum(map50[i-(window_size-1):i+1]) / window_size
                    rolling_avg.append(avg)
                
                # Plot rolling average
                plt.plot(epochs[window_size-1:], rolling_avg, 'b-', 
                         linewidth=1.5, label=f'{window_size}-epoch Moving Average')
            
            # Add best point annotation
            plt.axhline(y=best_map50, color='g', linestyle='--', alpha=0.5)
            plt.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
            plt.annotate(f'Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch})', 
                        xy=(best_epoch, best_map50),
                        xytext=(best_epoch*(0.9 if best_epoch > epochs[-1]/2 else 1.1), 
                                best_map50*(0.95 if best_map50 > sum(map50)/len(map50) else 1.05)),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                        fontsize=10)
            
            plt.legend()
            plt.tight_layout()
            
            if save_plots:
                plt.savefig(os.path.join(output_dir, 'map50_progress.png'), dpi=300)
                plt.close()
            else:
                plt.show()
            
            # Print final metrics with highlight on mAP@0.5
            print("\n" + "="*50)
            print("FINAL TRAINING METRICS")
            print("="*50)
            print(f"Precision (IoU=0.5): {precision[-1]:.4f}")
            print(f"Recall (IoU=0.5): {recall[-1]:.4f}")
            print(f"→ mAP@0.5 (IoU=0.5): {map50[-1]:.4f} ←")
            print(f"mAP@0.5:0.95: {map[-1]:.4f}")
            print(f"Best mAP@0.5: {best_map50:.4f} (Epoch {best_epoch})")
            print("="*50)
            
            return {
                'precision': precision[-1],
                'recall': recall[-1],
                'mAP50': map50[-1],
                'mAP50-95': map[-1],
                'best_mAP50': best_map50,
                'best_epoch': best_epoch
            }

In [36]:
os.path.join(os.getcwd(), 'yolo_results')

'/home/jason-server/Documents/amini_cocoa_contamination/yolo_results'

In [37]:
plot_yolo_performance(os.path.join(os.getcwd(), 'yolo_results'), output_dir='performance_plots')

In [38]:
test_image_dir = os.path.join(os.getcwd(), 'dataset', 'images', 'test')
model_path = os.path.join(os.getcwd(), 'yolov11_plant_detection', 'yolov11s_map50_focused', 'weights', 'best.pt') 
# Run inference on test images
run_inference(model_path, test_image_dir=test_image_dir)



WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1626 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_A16nzu.jpg: 640x480 1 healthy, 59.7ms
image 2/1626 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_A1Euyz.jpg: 640x480 2 anthracnoses, 7.0ms
image 3/1626 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_A1HcV0.jpeg: 640x480 1 healthy, 1 anthracnose, 6.9ms
image 4/1626 /home/jason-server/Documents/amini_co

KeyboardInterrupt: 

In [39]:
from ultralytics import YOLO
from tqdm.notebook import tqdm
model_path = os.path.join(os.getcwd(), 'yolov11_plant_detection', 'yolov11s_map50_focused', 'weights', 'best.pt') 

# Load the trained YOLO model
model = YOLO(model_path)

# Path to the test images directory
test_dir_path = TEST_IMAGES_DIR

# Get a list of all image files in the test directory
image_files = os.listdir(test_dir_path)

# Initialize an empty list to store the results for all images
all_data = []

# Initialize an empty list to store the results for all images
all_data = []

# Iterate through each image in the directory
for image_file in tqdm(image_files):
    # Full path to the image
    img_path = os.path.join(test_dir_path, image_file)

    # Make predictions on the image
    results = model(img_path)

    # Extract bounding boxes, confidence scores, and class labels
    boxes = results[0].boxes.xyxy.tolist() if results[0].boxes else []  # Bounding boxes in xyxy format
    classes = results[0].boxes.cls.tolist() if results[0].boxes else []  # Class indices
    confidences = results[0].boxes.conf.tolist() if results[0].boxes else []  # Confidence scores
    names = results[0].names  # Class names dictionary

    if boxes:  # If detections are found
        for box, cls, conf in zip(boxes, classes, confidences):
            x1, y1, x2, y2 = box
            detected_class = names[int(cls)]  # Get the class name from the names dictionary

            # Add the result to the all_data list
            all_data.append({
                'Image_ID': str(image_file),
                'class': detected_class,
                'confidence': conf,
                'ymin': y1,
                'xmin': x1,
                'ymax': y2,
                'xmax': x2
            })
    else:  # If no objects are detected
        all_data.append({
            'Image_ID': str(image_file),
            'class': "None",
            'confidence': None,
            'ymin': None,
            'xmin': None,
            'ymax': None,
            'xmax': None
        })

# Convert the list to a DataFrame for all images
sub = pd.DataFrame(all_data)

# Create submission file to be uploaded to Zindi for scoring
sub.to_csv(f'{INPUT_DATA_DIR / "YOLO_100epochs_BenchmarkSubmission_run14.csv"}', index = False)

  0%|          | 0/1626 [00:00<?, ?it/s]


image 1/1 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_zSiC0w.jpg: 640x480 1 healthy, 1 anthracnose, 7.6ms
Speed: 2.6ms preprocess, 7.6ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_kvMeX2.JPG: 640x480 (no detections), 7.0ms
Speed: 2.5ms preprocess, 7.0ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_EBhZzJ.jpeg: 640x640 1 anthracnose, 7.1ms
Speed: 3.2ms preprocess, 7.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID_p4aJv5.jpg: 640x480 1 cssvd, 7.6ms
Speed: 1.8ms preprocess, 7.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/jason-server/Documents/amini_cocoa_contamination/dataset/images/test/ID

In [40]:
sub

,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,ID_zSiC0w.jpg,healthy,0.738884,985.388550,541.326965,2929.503906,1659.822388
1,ID_zSiC0w.jpg,anthracnose,0.487250,949.585510,1356.254761,1730.766357,2448.000000
2,ID_kvMeX2.JPG,None,NaN,NaN,NaN,NaN,NaN
3,ID_EBhZzJ.jpeg,anthracnose,0.765557,33.015591,325.651093,3013.001953,2218.636963
4,ID_p4aJv5.jpg,cssvd,0.904809,235.863342,0.352043,1052.319336,434.581512
...,...,...,...,...,...,...,...
2857,ID_GeExO3.jpeg,None,NaN,NaN,NaN,NaN,NaN
2858,ID_xjnuvj.jpg,healthy,0.798862,45.639420,103.159744,356.133820,316.373352
2859,ID_xjnuvj.jpg,healthy,0.698474,0.046417,180.462189,102.207100,413.475525
2860,ID_xjnuvj.jpg,healthy,0.257210,42.518841,1.888914,169.696335,101.950920


# Adding normalization

In [1]:
def train_yolov11_full_dataset(output_dir='yolo_dataset', model_name_path='yolov11s.pt',
                         model_size='s', epochs=100, batch_size=16, image_size=640, cls_gain=0.5,
                         warmup_epochs=3.0, initial_lr=0.01, pretrained=True,
                         train_val_split=0.8, random_state=42, normalize_boxes=True):
    """
    Train a YOLOv11 model on the full dataset with a single train/val split and normalized bounding boxes.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        model_name_path: path to the base model
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: number of training epochs
        batch_size: batch size
        image_size: input image size
        cls_gain: classification loss weight
        warmup_epochs: number of warmup epochs
        initial_lr: initial learning rate
        pretrained: whether to use pretrained weights
        train_val_split: ratio of data to use for training (1 - train_val_split for validation)
        random_state: random seed for reproducibility
        normalize_boxes: whether to normalize bounding boxes (0-1 scale)
        
    Returns:
        Dictionary containing training results
    """
    import os
    import shutil
    import yaml
    import json
    import numpy as np
    from datetime import datetime
    from sklearn.model_selection import train_test_split
    from ultralytics import YOLO
    
    print(f"\n{'='*50}")
    print(f"Training YOLOv11 on full dataset with normalized bounding boxes")
    print(f"{'='*50}")
    
    # Create directories for train/val split
    for split in ['train', 'val']:
        for subdir in ['images', 'labels']:
            fold_dir = os.path.join(output_dir, 'full_dataset', split, subdir)
            os.makedirs(fold_dir, exist_ok=True)
    
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Split data
    train_images, val_images = train_test_split(
        all_images, 
        train_size=train_val_split, 
        random_state=random_state
    )
    
    print(f"Dataset split: {len(train_images)} training images, {len(val_images)} validation images")
    
    # Process and copy data
    for split, images in [('train', train_images), ('val', val_images)]:
        for img_file in images:
            # Get corresponding label file
            label_file = os.path.splitext(img_file)[0] + '.txt'
            
            # Copy image
            src_img = os.path.join(output_dir, 'images', img_file)
            dst_img = os.path.join(output_dir, 'full_dataset', split, 'images', img_file)
            shutil.copy(src_img, dst_img)
            
            # Process and copy label
            src_label = os.path.join(output_dir, 'labels', label_file)
            dst_label = os.path.join(output_dir, 'full_dataset', split, 'labels', label_file)
            
            if os.path.exists(src_label):
                if normalize_boxes:
                    # Read and normalize bounding boxes if needed
                    with open(src_label, 'r') as f:
                        lines = f.readlines()
                    
                    # Load corresponding image to get dimensions
                    from PIL import Image
                    img = Image.open(src_img)
                    img_width, img_height = img.size
                    
                    normalized_lines = []
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            # YOLO format: class_id center_x center_y width height
                            class_id = parts[0]
                            x_center, y_center, width, height = map(float, parts[1:5])
                            
                            # Check if already normalized (0-1)
                            if max(x_center, y_center, width, height) > 1.0:
                                # Convert absolute coordinates to normalized (0-1)
                                x_center /= img_width
                                y_center /= img_height
                                width /= img_width
                                height /= img_height
                            
                            normalized_line = f"{class_id} {x_center} {y_center} {width} {height}"
                            if len(parts) > 5:  # Preserve additional attributes if any
                                normalized_line += " " + " ".join(parts[5:])
                            normalized_lines.append(normalized_line + "\n")
                    
                    # Write normalized labels
                    with open(dst_label, 'w') as f:
                        f.writelines(normalized_lines)
                else:
                    # Just copy the label file as is
                    shutil.copy(src_label, dst_label)
    
    # Get class names
    classes_path = os.path.join(output_dir, 'classes.txt')
    if os.path.exists(classes_path):
        with open(classes_path, 'r') as f:
            classes = [line.strip() for line in f.readlines()]
    else:
        # Try to infer class count from labels
        class_ids = set()
        labels_dir = os.path.join(output_dir, 'labels')
        for label_file in os.listdir(labels_dir):
            if label_file.endswith('.txt'):
                with open(os.path.join(labels_dir, label_file), 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            class_ids.add(int(parts[0]))
        
        # Create generic class names
        classes = [f"class_{i}" for i in range(max(class_ids) + 1)] if class_ids else ["object"]
    
    # Create YAML configuration
    config = {
        'path': os.path.abspath(os.path.join(output_dir, 'full_dataset')),
        'train': 'train/images',
        'val': 'val/images',
        'nc': len(classes),
        'names': classes
    }
    
    yaml_path = os.path.join(output_dir, 'dataset_full.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"Created YAML configuration file: {yaml_path}")
    
    # Train model
    model = YOLO(model_name_path)
    
    # Create project name
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    project_name = f'yolov11_full_{model_size}'
    run_name = f'normalized_boxes_{timestamp}' if normalize_boxes else f'standard_{timestamp}'
    
    # Train model
    results = model.train(
        data=yaml_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=image_size,
        patience=epochs // 2,  # Early stopping patience
        cache=True,
        project=project_name,
        name=run_name,
        save=True,
        pretrained=pretrained,
        verbose=True,
        val=True,
        save_period=10,
        lr0=initial_lr,
        lrf=0.01,
        warmup_epochs=warmup_epochs,
        box=7.5,
        cls=cls_gain,
        cos_lr=True,
        close_mosaic=10,
        amp=True,
        plots=True
    )
    
    # Extract metrics
    metrics = model.metrics.results_dict
    map50 = metrics.get('metrics/mAP50(B)', 0.0)
    
    # Store training results
    train_results = {
        'timestamp': timestamp,
        'model_size': model_size,
        'epochs': epochs,
        'batch_size': batch_size,
        'map50': float(map50),
        'train_images': len(train_images),
        'val_images': len(val_images),
        'normalized_boxes': normalize_boxes,
        'yaml_path': yaml_path,
        'model_path': os.path.join(project_name, run_name, 'weights', 'best.pt')
    }
    
    # Save training results
    results_path = os.path.join(output_dir, f'train_results_{timestamp}.json')
    with open(results_path, 'w') as f:
        json.dump(train_results, f, indent=2)
    
    print(f"\n{'='*50}")
    print(f"Training Results")
    print(f"{'='*50}")
    print(f"mAP50: {map50:.4f}")
    print(f"Trained with normalized bounding boxes: {normalize_boxes}")
    print(f"Model saved to: {train_results['model_path']}")
    print(f"Results saved to: {results_path}")
    
    return train_results

In [ ]:
# Train with normalized bounding boxes
results = train_yolov11_full_dataset(
    output_dir='my_dataset',
    model_name_path='yolov11s.pt',
    model_size='s',
    epochs=100,
    batch_size=16,
    normalize_boxes=True,
    train_val_split=0.8
)

# Or train with original bounding boxes
results = train_yolov11_full_dataset(
    output_dir='my_dataset',
    model_name_path='yolov11s.pt',
    normalize_boxes=False
)

# Adding k fold validation, normalization of bounding boxes and less epochs

In [41]:
import os
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split, KFold
from PIL import Image
import shutil
import yaml
import torch
import json
from datetime import datetime

def k_fold_cross_validation(output_dir='yolo_dataset', n_splits=5, random_state=42):
    """
    Create k-fold cross-validation splits for YOLO training.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        n_splits: number of folds
        random_state: random seed for reproducibility
        
    Returns:
        List of dictionaries containing train and validation indices for each fold
    """
    print(f"Creating {n_splits}-fold cross-validation splits...")
    
    # Get all image filenames
    image_dir = os.path.join(output_dir, 'images')
    all_images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Initialize KFold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Create fold directories
    for fold_idx in range(n_splits):
        for split in ['train', 'val']:
            for subdir in ['images', 'labels']:
                fold_dir = os.path.join(output_dir, f'fold{fold_idx+1}', split, subdir)
                os.makedirs(fold_dir, exist_ok=True)
    
    # Generate folds
    fold_info = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_images)):
        train_images = [all_images[i] for i in train_idx]
        val_images = [all_images[i] for i in val_idx]
        
        fold_info.append({
            'fold': fold_idx + 1,
            'train_images': len(train_images),
            'val_images': len(val_images)
        })
        
        # Create train and val splits for this fold
        for split, images in [('train', train_images), ('val', val_images)]:
            for img_file in images:
                # Get corresponding label file
                label_file = os.path.splitext(img_file)[0] + '.txt'
                
                # Copy image
                src_img = os.path.join(output_dir, 'images', img_file)
                dst_img = os.path.join(output_dir, f'fold{fold_idx+1}', split, 'images', img_file)
                shutil.copy(src_img, dst_img)
                
                # Copy label
                src_label = os.path.join(output_dir, 'labels', label_file)
                dst_label = os.path.join(output_dir, f'fold{fold_idx+1}', split, 'labels', label_file)
                if os.path.exists(src_label):  # Some images might not have annotations
                    shutil.copy(src_label, dst_label)
    
    # Print fold information
    print(f"K-fold splits created:")
    for fold in fold_info:
        print(f"  Fold {fold['fold']}: {fold['train_images']} train, {fold['val_images']} validation images")
    
    return fold_info

def create_fold_yaml_configs(output_dir='yolo_dataset', classes=None, n_splits=5):
    """
    Create YAML configuration files for each fold.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        classes: list of class names
        n_splits: number of folds
        
    Returns:
        List of paths to the YAML configuration files
    """
    # If classes not provided, load from classes.txt
    if classes is None:
        classes_path = os.path.join(output_dir, 'classes.txt')
        if os.path.exists(classes_path):
            with open(classes_path, 'r') as f:
                classes = [line.strip() for line in f.readlines()]
        else:
            raise ValueError("Classes not provided and classes.txt not found.")
    
    yaml_paths = []
    
    for fold_idx in range(1, n_splits + 1):
        config = {
            'path': os.path.abspath(os.path.join(output_dir, f'fold{fold_idx}')),
            'train': 'train/images',
            'val': 'val/images',
            'nc': len(classes),
            'names': classes
        }
        
        yaml_path = os.path.join(output_dir, f'dataset_fold{fold_idx}.yaml')
        with open(yaml_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False)
        
        yaml_paths.append(yaml_path)
    
    print(f"Created {n_splits} YAML configuration files for cross-validation")
    return yaml_paths

def train_kfold_yolov11(output_dir='yolo_dataset', model_name_path='yolov11s.pt', n_splits=5, 
                        model_size='s', epochs=100, batch_size=16, image_size=640, cls_gain=0.5, 
                        warmup_epochs=3.0, initial_lr=0.01, pretrained=True):
    """
    Train a YOLOv11 model with k-fold cross-validation.
    
    Args:
        output_dir: directory containing the YOLO format dataset
        model_name_path: path to the base model
        n_splits: number of folds
        model_size: YOLOv11 model size ('n', 's', 'm', 'l', 'x')
        epochs: number of training epochs
        batch_size: batch size
        image_size: input image size
        cls_gain: classification loss weight
        warmup_epochs: number of warmup epochs
        initial_lr: initial learning rate
        pretrained: whether to use pretrained weights
        
    Returns:
        Dictionary containing cross-validation results
    """
    # Import YOLO
    from ultralytics import YOLO
    
    # Create k-fold splits
    fold_info = k_fold_cross_validation(output_dir=output_dir, n_splits=n_splits)
    
    # Create YAML configs for each fold
    yaml_paths = create_fold_yaml_configs(output_dir=output_dir, n_splits=n_splits)
    
    # Train model for each fold
    cv_results = {
        'folds': [],
        'mean_map50': 0.0,
        'std_map50': 0.0,
        'best_fold': 0,
        'best_map50': 0.0,
        'model_size': model_size,
        'epochs': epochs,
        'batch_size': batch_size,
        'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S")
    }
    
    map50_scores = []
    
    for fold_idx, yaml_path in enumerate(yaml_paths):
        print(f"\n{'='*50}")
        print(f"Training fold {fold_idx+1}/{n_splits}")
        print(f"{'='*50}")
        
        # Load model for this fold
        model = YOLO(model_name_path)
        
        # Create project name for this fold
        project_name = f'yolov11_cv_{model_size}'
        run_name = f'fold{fold_idx+1}_of_{n_splits}'
        
        # Train model
        results = model.train(
            data=yaml_path,
            epochs=epochs,
            batch=batch_size,
            imgsz=image_size,
            patience=epochs // 2,  # Early stopping patience
            cache=True,
            project=project_name,
            name=run_name,
            save=True,
            pretrained=pretrained,
            verbose=True,
            val=True,
            save_period=10,
            lr0=initial_lr,
            lrf=0.01,
            warmup_epochs=warmup_epochs,
            box=7.5,
            cls=cls_gain,
            cos_lr=True,
            close_mosaic=10,
            amp=True,
            plots=True
        )
        
        # Extract metrics
        metrics = model.metrics.results_dict
        map50 = metrics.get('metrics/mAP50(B)', 0.0)
        map50_scores.append(map50)
        
        # Store fold results
        fold_result = {
            'fold': fold_idx + 1,
            'map50': map50,
            'yaml_path': yaml_path,
            'model_path': os.path.join(project_name, run_name, 'weights', 'best.pt')
        }
        cv_results['folds'].append(fold_result)
        
        print(f"Fold {fold_idx+1} mAP50: {map50:.4f}")
    
    # Calculate cross-validation statistics
    mean_map50 = np.mean(map50_scores)
    std_map50 = np.std(map50_scores)
    best_fold_idx = np.argmax(map50_scores)
    best_map50 = map50_scores[best_fold_idx]
    
    cv_results['mean_map50'] = float(mean_map50)
    cv_results['std_map50'] = float(std_map50)
    cv_results['best_fold'] = int(best_fold_idx + 1)
    cv_results['best_map50'] = float(best_map50)
    
    # Save cross-validation results
    results_path = os.path.join(output_dir, f'cv_results_{cv_results["timestamp"]}.json')
    with open(results_path, 'w') as f:
        json.dump(cv_results, f, indent=2)
    
    print(f"\n{'='*50}")
    print(f"Cross-Validation Results")
    print(f"{'='*50}")
    print(f"Mean mAP50: {mean_map50:.4f}")
    print(f"Standard Deviation: {std_map50:.4f}")
    print(f"Best Fold: {best_fold_idx + 1} (mAP50: {best_map50:.4f})")
    print(f"Results saved to: {results_path}")
    
    return cv_results

def ensemble_kfold_models(cv_results, output_dir='yolo_dataset', test_images_dir=None, 
                          conf_threshold=0.25, iou_threshold=0.45):
    """
    Create an ensemble of k-fold models for inference.
    
    Args:
        cv_results: cross-validation results from train_kfold_yolov11
        output_dir: directory containing the YOLO format dataset
        test_images_dir: directory containing test images (if None, uses validation set from best fold)
        conf_threshold: confidence threshold for detections
        iou_threshold: IoU threshold for NMS
        
    Returns:
        Ensemble prediction results
    """
    from ultralytics import YOLO
    
    print(f"\n{'='*50}")
    print(f"Creating Model Ensemble from K-Fold Models")
    print(f"{'='*50}")
    
    # Load all models
    models = []
    for fold_info in cv_results['folds']:
        model_path = fold_info['model_path']
        if os.path.exists(model_path):
            print(f"Loading model from fold {fold_info['fold']}: {model_path}")
            model = YOLO(model_path)
            models.append(model)
        else:
            print(f"WARNING: Model not found for fold {fold_info['fold']}: {model_path}")
    
    if len(models) == 0:
        raise ValueError("No models found for ensemble.")
    
    # Determine test images directory
    if test_images_dir is None:
        best_fold = cv_results['best_fold']
        best_fold_yaml = cv_results['folds'][best_fold - 1]['yaml_path']
        
        # Load YAML config to get validation data path
        with open(best_fold_yaml, 'r') as f:
            config = yaml.safe_load(f)
            base_path = config['path']
            val_path = os.path.join(base_path, config['val'])
            test_images_dir = val_path
    
    print(f"Running ensemble inference on: {test_images_dir}")
    
    # Create output directory for ensemble predictions
    ensemble_dir = os.path.join(output_dir, f'ensemble_{cv_results["timestamp"]}')
    os.makedirs(ensemble_dir, exist_ok=True)
    
    # Run predictions with each model
    all_predictions = []
    
    for i, model in enumerate(models):
        print(f"Running predictions with model {i+1}/{len(models)}")
        results = model.predict(
            source=test_images_dir,
            conf=conf_threshold,
            iou=iou_threshold,
            save=False,
            save_txt=False,
            verbose=False
        )
        all_predictions.append(results)
    
    # Combine predictions
    ensemble_predictions = []
    processed_images = set()
    
    print("Combining predictions from all models...")
    for model_idx, model_results in enumerate(all_predictions):
        for result in model_results:
            img_name = os.path.basename(result.path)
            
            # Extract predictions
            if len(result.boxes) > 0:
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = box.conf.item()
                    cls = int(box.cls.item())
                    class_name = models[model_idx].names[cls]
                    
                    ensemble_predictions.append({
                        'image_name': img_name,
                        'class': class_name,
                        'confidence': conf,
                        'xmin': x1,
                        'ymin': y1,
                        'xmax': x2,
                        'ymax': y2,
                        'model': model_idx + 1
                    })
                    
                    processed_images.add(img_name)
    
    # Create DataFrame and save
    if ensemble_predictions:
        pred_df = pd.DataFrame(ensemble_predictions)
        predictions_path = os.path.join(ensemble_dir, 'ensemble_predictions.csv')
        pred_df.to_csv(predictions_path, index=False)
        
        # Also save summarized predictions
        image_count = len(processed_images)
        prediction_count = len(ensemble_predictions)
        model_count = len(models)
        
        summary = {
            'timestamp': cv_results['timestamp'],
            'models_used': model_count,
            'images_processed': image_count,
            'total_predictions': prediction_count,
            'average_predictions_per_image': prediction_count / max(1, image_count),
            'conf_threshold': conf_threshold,
            'iou_threshold': iou_threshold
        }
        
        summary_path = os.path.join(ensemble_dir, 'ensemble_summary.json')
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"Ensemble predictions saved to: {predictions_path}")
        print(f"Processed {image_count} images with {prediction_count} total detections")
    else:
        print("No predictions generated by the ensemble.")
    
    return ensemble_predictions

In [43]:
output_dir_to_use = "yolo_dataset_kfold_normed"
# Step 1: Process your dataset (your existing code)
classes = process_dataset(train_df, output_dir=output_dir_to_use)

# Step 2: Instead of splitting the dataset, run k-fold cross-validation
k_fold_cross_validation(output_dir=output_dir_to_use, n_splits=5)


Processed 5529 images with 9792 annotations.
Creating 5-fold cross-validation splits...
K-fold splits created:
  Fold 1: 4197 train, 1050 validation images
  Fold 2: 4197 train, 1050 validation images
  Fold 3: 4198 train, 1049 validation images
  Fold 4: 4198 train, 1049 validation images
  Fold 5: 4198 train, 1049 validation images


[{'fold': 1, 'train_images': 4197, 'val_images': 1050},
 {'fold': 2, 'train_images': 4197, 'val_images': 1050},
 {'fold': 3, 'train_images': 4198, 'val_images': 1049},
 {'fold': 4, 'train_images': 4198, 'val_images': 1049},
 {'fold': 5, 'train_images': 4198, 'val_images': 1049}]

In [44]:

# Step 3: Train models for all folds
cv_results = train_kfold_yolov11(
    output_dir=output_dir_to_use,
    model_name_path=pretrained_yolo_path,  # or your pretrained model path
    n_splits=5,
    epochs=20,
    batch_size=64,
    cls_gain=0.5,
    pretrained=True  # This allows backpropagation on all weights
)


Creating 5-fold cross-validation splits...
K-fold splits created:
  Fold 1: 4197 train, 1050 validation images
  Fold 2: 4197 train, 1050 validation images
  Fold 3: 4198 train, 1049 validation images
  Fold 4: 4198 train, 1049 validation images
  Fold 5: 4198 train, 1049 validation images
Created 5 YAML configuration files for cross-validation

Training fold 1/5
New https://pypi.org/project/ultralytics/8.3.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
engine/trainer: task=detect, mode=train, model=plant-leaf-detection-and-classification/best.pt, data=yolo_dataset_kfold_normed/dataset_fold1.yaml, epochs=20, time=None, patience=10, batch=64, imgsz=640, save=True, save_period=10, cache=True, device=None, workers=8, project=yolov11_cv_s, name=fold1_of_5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=Tru

train: Scanning /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/labels... 4197 images, 0 backgrounds, 686 corrupt: 100%|██████████| 4197/4197 [00:11<00:00, 379.42it/s]

train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/images/ID_AC3jGA.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2288      1.0926]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/images/ID_AIHFIo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3267      1.1366]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/images/ID_AJD939.jpg: corrupt JPEG restored and saved
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/images/ID_AK7dFo.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.3218      1.2725]
train: WARNING ⚠️ /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/images/ID_AMshRT.jpeg:

train: New cache created: /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/train/labels.cache
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


train: Caching images (3.1GB RAM): 100%|██████████| 3511/3511 [00:19<00:00, 181.17it/s]
val: Scanning /home/jason-server/Documents/amini_cocoa_contamination/yolo_dataset_kfold_normed/fold1/val/labels... 1050 images, 0 backgrounds, 195 corrupt: 100%|██████████| 1050/1050 [00:03<00:00, 276.53it/s]

...

      20/20      18.1G     0.9877     0.8369      1.442         54        640: 100%|██████████| 55/55 [00:16<00:00,  3.27it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:02<00:00,  2.40it/s]

                   all        878       1592      0.711      0.633      0.706      0.467



20 epochs completed in 0.116 hours.
Optimizer stripped from yolov11_cv_s/fold5_of_5/weights/last.pt, 22.5MB
Optimizer stripped from yolov11_cv_s/fold5_of_5/weights/best.pt, 22.5MB

Validating yolov11_cv_s/fold5_of_5/weights/best.pt...
Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3090, 24253MiB)
Model summary (fused): 72 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:04<00:00,  1.70it/s]


                   all        878       1592        0.7      0.655      0.711      0.471
               healthy        243        634      0.633      0.632      0.665       0.42
           anthracnose        208        335      0.742      0.661      0.744      0.497
                 cssvd        427        623      0.724      0.673      0.723      0.496
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to yolov11_cv_s/fold5_of_5
Fold 5 mAP50: 0.7108

Cross-Validation Results
Mean mAP50: 0.7035
Standard Deviation: 0.0091
Best Fold: 5 (mAP50: 0.7108)
Results saved to: yolo_dataset_kfold_normed/cv_results_20250327_004046.json


In [45]:
cv_results

{'folds': [{'fold': 1,
   'map50': 0.7097250065108804,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold1.yaml',
   'model_path': 'yolov11_cv_s/fold1_of_5/weights/best.pt'},
  {'fold': 2,
   'map50': 0.7014380407234279,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold2.yaml',
   'model_path': 'yolov11_cv_s/fold2_of_5/weights/best.pt'},
  {'fold': 3,
   'map50': 0.6866092215552629,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold3.yaml',
   'model_path': 'yolov11_cv_s/fold3_of_5/weights/best.pt'},
  {'fold': 4,
   'map50': 0.7090942802995972,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold4.yaml',
   'model_path': 'yolov11_cv_s/fold4_of_5/weights/best.pt'},
  {'fold': 5,
   'map50': 0.7107842119610798,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold5.yaml',
   'model_path': 'yolov11_cv_s/fold5_of_5/weights/best.pt'}],
 'mean_map50': 0.7035301522100496,
 'std_map50': 0.009084623809200736,
 'best_fold': 5,
 'best_map50': 0.7107842119610798,
 'model_size

In [6]:
cv_results = {'folds': [{'fold': 1,
   'map50': 0.7097250065108804,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold1.yaml',
   'model_path': 'yolov11_cv_s/fold1_of_5/weights/best.pt'},
  {'fold': 2,
   'map50': 0.7014380407234279,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold2.yaml',
   'model_path': 'yolov11_cv_s/fold2_of_5/weights/best.pt'},
  {'fold': 3,
   'map50': 0.6866092215552629,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold3.yaml',
   'model_path': 'yolov11_cv_s/fold3_of_5/weights/best.pt'},
  {'fold': 4,
   'map50': 0.7090942802995972,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold4.yaml',
   'model_path': 'yolov11_cv_s/fold4_of_5/weights/best.pt'},
  {'fold': 5,
   'map50': 0.7107842119610798,
   'yaml_path': 'yolo_dataset_kfold_normed/dataset_fold5.yaml',
   'model_path': 'yolov11_cv_s/fold5_of_5/weights/best.pt'}],
 'mean_map50': 0.7035301522100496,
 'std_map50': 0.009084623809200736,
 'best_fold': 5,
 'best_map50': 0.7107842119610798,
 'model_size': 's',
 'epochs': 20,
 'batch_size': 64,
 'timestamp': '20250327_004046'}

In [7]:
TEST_IMAGES_DIR

PosixPath('dataset/images/test')

# Ensemble the predictions of the 5 seperate yolo models

In [25]:
def ensemble_kfold_models_optimized(cv_results, output_dir='yolo_dataset', test_images_dir=None, 
                               conf_threshold=0.25, iou_threshold=0.45):
    """
    Create an ensemble of k-fold models for inference, loading only one model at a time to save memory.
    
    Args:
        cv_results: cross-validation results from train_kfold_yolov11
        output_dir: directory containing the YOLO format dataset
        test_images_dir: directory containing test images (if None, uses validation set from best fold)
        conf_threshold: confidence threshold for detections
        iou_threshold: IoU threshold for NMS
        
    Returns:
        Ensemble prediction results
    """
    from ultralytics import YOLO
    import os
    import pandas as pd
    import json
    from datetime import datetime
    import gc
    import torch
    
    print(f"\n{'='*50}")
    print(f"Creating Memory-Optimized Model Ensemble from K-Fold Models")
    print(f"{'='*50}")
    
    # Get model paths
    model_paths = []
    for fold_info in cv_results['folds']:
        model_path = fold_info['model_path']
        if os.path.exists(model_path):
            print(f"Found model from fold {fold_info['fold']}: {model_path}")
            model_paths.append((fold_info['fold'], model_path))
        else:
            print(f"WARNING: Model not found for fold {fold_info['fold']}: {model_path}")
    
    if len(model_paths) == 0:
        raise ValueError("No models found for ensemble.")
    
    # Determine test images directory
    if test_images_dir is None:
        best_fold = cv_results['best_fold']
        best_fold_yaml = cv_results['folds'][best_fold - 1]['yaml_path']
        
        # Load YAML config to get validation data path
        with open(best_fold_yaml, 'r') as f:
            import yaml
            config = yaml.safe_load(f)
            base_path = config['path']
            val_path = os.path.join(base_path, config['val'])
            test_images_dir = val_path
    
    print(f"Running ensemble inference on: {test_images_dir}")
    
    # Create output directory for ensemble predictions
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if "timestamp" not in cv_results else cv_results["timestamp"]
    ensemble_dir = os.path.join(output_dir, f'ensemble_{timestamp}')
    os.makedirs(ensemble_dir, exist_ok=True)
    
    # Get list of all test images to ensure we process each image once
    import glob
    test_image_paths = glob.glob(os.path.join(test_images_dir, '*.jpg')) + \
                        glob.glob(os.path.join(test_images_dir, '*.jpeg')) + \
                        glob.glob(os.path.join(test_images_dir, '*.png'))
    
    # Extract base filenames to track which images have been processed
    all_image_names = [os.path.basename(path) for path in test_image_paths]
    print(f"Found {len(all_image_names)} images in test directory")
    
    # Prepare storage for all predictions
    ensemble_predictions = []
    processed_images = set()
    
    # Process each model individually
    for fold_idx, model_path in model_paths:
        print(f"\nRunning predictions with model from fold {fold_idx}")
        
        # Load model
        model = YOLO(model_path)
        
        # Run predictions with current model
        results = model.predict(
            source=test_images_dir,
            conf=conf_threshold,
            iou=iou_threshold,
            save=False,
            save_txt=False,
            verbose=False
        )
        
        # Extract predictions from current model results
        print(f"Processing predictions from fold {fold_idx}...")
        class_names = model.names  # Get class names from the model
        
        for result in results:
            img_name = os.path.basename(result.path)
            
            # Extract predictions
            if len(result.boxes) > 0:
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = box.conf.item()
                    cls = int(box.cls.item())
                    class_name = class_names[cls]
                    
                    ensemble_predictions.append({
                        'image_name': img_name,
                        'class': class_name,
                        'confidence': conf,
                        'xmin': x1,
                        'ymin': y1,
                        'xmax': x2,
                        'ymax': y2,
                        'model_fold': fold_idx
                    })
                    
                    processed_images.add(img_name)
        
        # Clear GPU memory for the current model
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        print(f"Completed processing model from fold {fold_idx}")
    
    # Create DataFrame and save results
    if ensemble_predictions:
        pred_df = pd.DataFrame(ensemble_predictions)
        
        # Create a DataFrame with all images (including those without predictions)
        all_images_df = pd.DataFrame({'image_name': all_image_names})
        
        # Check which images have no predictions
        images_with_predictions = set(pred_df['image_name'].unique())
        missing_images = set(all_image_names) - images_with_predictions
        
        # If there are images without predictions, add placeholder rows with NaN values
        if missing_images:
            print(f"Adding {len(missing_images)} images with no predictions to the results")
            missing_rows = []
            for img_name in missing_images:
                # Add a row with NaN values for each missing image
                missing_row = {
                    'image_name': img_name,
                    'class': None,
                    'confidence': np.nan,
                    'xmin': np.nan,
                    'ymin': np.nan,
                    'xmax': np.nan,
                    'ymax': np.nan,
                    'model_fold': np.nan
                }
                missing_rows.append(missing_row)
            
            # Combine with existing predictions
            if missing_rows:
                missing_df = pd.DataFrame(missing_rows)
                pred_df = pd.concat([pred_df, missing_df], ignore_index=True)
        
        # Save the complete prediction results
        predictions_path = os.path.join(ensemble_dir, 'ensemble_predictions.csv')
        pred_df.to_csv(predictions_path, index=False)
        
        # Generate summary statistics and save
        image_count = len(all_image_names)
        images_with_detections = len(images_with_predictions)
        prediction_count = len(ensemble_predictions)
        model_count = len(model_paths)
        
        # Group by image and class for ensemble consensus
        pred_df['detection_count'] = 1
        ensemble_summary = pred_df.groupby(['image_name', 'class']).agg({
            'confidence': ['mean', 'max', 'count'],
            'detection_count': 'sum'
        }).reset_index()
        
        # Flatten the column hierarchy
        ensemble_summary.columns = ['_'.join(col).strip('_') for col in ensemble_summary.columns.values]
        
        # Save ensemble summary
        ensemble_summary_path = os.path.join(ensemble_dir, 'ensemble_detection_summary.csv')
        ensemble_summary.to_csv(ensemble_summary_path, index=False)
        
        # Overall summary
        summary = {
            'timestamp': timestamp,
            'models_used': model_count,
            'images_processed': image_count,
            'images_with_detections': images_with_detections,
            'images_without_detections': len(missing_images),
            'total_predictions': prediction_count,
            'average_predictions_per_detected_image': prediction_count / max(1, images_with_detections),
            'average_predictions_per_all_images': prediction_count / max(1, image_count),
            'conf_threshold': conf_threshold,
            'iou_threshold': iou_threshold,
            'memory_optimized': True
        }
        
        summary_path = os.path.join(ensemble_dir, 'ensemble_summary.json')
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\n{'='*50}")
        print(f"Ensemble Results")
        print(f"{'='*50}")
        print(f"Ensemble predictions saved to: {predictions_path}")
        print(f"Detection summary saved to: {ensemble_summary_path}")
        print(f"Processed {image_count} total images:")
        print(f"  - {images_with_detections} images with detections")
        print(f"  - {len(missing_images)} images with no detections (included in results with NaN values)")
        print(f"Total of {prediction_count} detections across all images and models")
        print(f"Used {model_count} models in the ensemble")
    else:
        print("No predictions generated by the ensemble.")
    
    return ensemble_predictions


def perform_ensemble_post_processing(predictions_path, output_dir=None, 
                                    confidence_threshold=0.5, 
                                    model_agreement_threshold=0.5,
                                    nms_iou_threshold=0.5):
    """
    Perform post-processing on ensemble predictions to combine overlapping detections.
    
    Args:
        predictions_path: path to the ensemble_predictions.csv file
        output_dir: directory to save post-processed results (defaults to same directory as predictions)
        confidence_threshold: minimum confidence threshold for final detections
        model_agreement_threshold: minimum fraction of models that must agree on a detection
        nms_iou_threshold: IoU threshold for non-maximum suppression
        
    Returns:
        DataFrame of post-processed predictions
    """
    import pandas as pd
    import numpy as np
    import os
    from datetime import datetime
    import json
    
    print(f"\n{'='*50}")
    print(f"Performing Ensemble Post-Processing")
    print(f"{'='*50}")
    
    # Load predictions
    pred_df = pd.read_csv(predictions_path)
    
    if len(pred_df) == 0:
        print("No predictions to process.")
        return pd.DataFrame()
    
    # Create output directory if not provided
    if output_dir is None:
        output_dir = os.path.dirname(predictions_path)
    
    # Count total number of models
    total_models = pred_df['model_fold'].nunique()
    
    # Group by image
    image_names = pred_df['image_name'].unique()
    
    # Prepare storage for post-processed results
    final_predictions = []
    
    # Define NMS function (Non-Maximum Suppression)
    def non_max_suppression(boxes, scores, iou_threshold):
        """
        Apply non-maximum suppression to boxes with scores.
        
        Args:
            boxes: list of [xmin, ymin, xmax, ymax]
            scores: confidence scores
            iou_threshold: IoU threshold for suppression
            
        Returns:
            Indices of boxes to keep
        """
        if len(boxes) == 0:
            return []
            
        # Convert to numpy arrays
        boxes = np.array(boxes)
        scores = np.array(scores)
        
        # Get indices of boxes sorted by score in descending order
        indices = np.argsort(scores)[::-1]
        
        keep = []
        while indices.size > 0:
            # Get index of highest scoring box
            i = indices[0]
            keep.append(i)
            
            # Calculate IoU of the highest scoring box with all other boxes
            ious = []
            for j in indices[1:]:
                # Calculate intersection area
                xx1 = max(boxes[i][0], boxes[j][0])
                yy1 = max(boxes[i][1], boxes[j][1])
                xx2 = min(boxes[i][2], boxes[j][2])
                yy2 = min(boxes[i][3], boxes[j][3])
                
                w = max(0, xx2 - xx1)
                h = max(0, yy2 - yy1)
                intersection = w * h
                
                # Calculate union area
                box1_area = (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1])
                box2_area = (boxes[j][2] - boxes[j][0]) * (boxes[j][3] - boxes[j][1])
                union = box1_area + box2_area - intersection
                
                iou = intersection / union if union > 0 else 0
                ious.append(iou)
            
            # Keep boxes with IoU less than threshold
            indices = indices[1:][np.array(ious) < iou_threshold]
            
        return keep
    
    # Process each image
    for img_name in image_names:
        img_preds = pred_df[pred_df['image_name'] == img_name]
        
        # Process each class separately
        for class_name in img_preds['class'].unique():
            class_preds = img_preds[img_preds['class'] == class_name]
            
            # Get boxes and scores
            boxes = class_preds[['xmin', 'ymin', 'xmax', 'ymax']].values.tolist()
            scores = class_preds['confidence'].values.tolist()
            model_ids = class_preds['model_fold'].values.tolist()
            
            # Apply NMS
            keep_indices = non_max_suppression(boxes, scores, nms_iou_threshold)
            
            # Process each cluster of detections
            for idx in keep_indices:
                box = boxes[idx]
                score = scores[idx]
                model_id = model_ids[idx]
                
                # Find overlapping detections from other models
                overlapping_models = set([model_id])
                overlapping_scores = [score]
                
                for i, other_box in enumerate(boxes):
                    if i != idx:
                        # Calculate IoU
                        xx1 = max(box[0], other_box[0])
                        yy1 = max(box[1], other_box[1])
                        xx2 = min(box[2], other_box[2])
                        yy2 = min(box[3], other_box[3])
                        
                        w = max(0, xx2 - xx1)
                        h = max(0, yy2 - yy1)
                        intersection = w * h
                        
                        box1_area = (box[2] - box[0]) * (box[3] - box[1])
                        box2_area = (other_box[2] - other_box[0]) * (other_box[3] - other_box[1])
                        union = box1_area + box2_area - intersection
                        
                        iou = intersection / union if union > 0 else 0
                        
                        if iou > nms_iou_threshold:
                            overlapping_models.add(model_ids[i])
                            overlapping_scores.append(scores[i])
                
                # Calculate model agreement
                model_agreement = len(overlapping_models) / total_models
                
                # Calculate weighted box coordinates
                if len(overlapping_scores) > 1:
                    # First collect all indices of overlapping boxes
                    overlapping_indices = [idx]
                    for j in range(len(boxes)):
                        if j != idx and model_ids[j] in overlapping_models:
                            overlapping_indices.append(j)
                    
                    # Now weights and indices have matching sizes
                    weights = np.array(overlapping_scores) / sum(overlapping_scores)
                    weighted_box = np.zeros(4)
                    
                    # Make sure we don't go out of bounds
                    for i in range(min(len(weights), len(overlapping_indices))):
                        other_idx = overlapping_indices[i]
                        weighted_box += weights[i] * np.array(boxes[other_idx])
                    
                    final_box = weighted_box.tolist()
                else:
                    final_box = box
                
                # Calculate average confidence
                avg_confidence = sum(overlapping_scores) / len(overlapping_scores)
                
                # Add to final predictions if meets thresholds
                if avg_confidence >= confidence_threshold and model_agreement >= model_agreement_threshold:
                    final_predictions.append({
                        'image_name': img_name,
                        'class': class_name,
                        'confidence': avg_confidence,
                        'xmin': final_box[0],
                        'ymin': final_box[1],
                        'xmax': final_box[2],
                        'ymax': final_box[3],
                        'model_agreement': model_agreement,
                        'models_detected': len(overlapping_models),
                        'total_models': total_models
                    })
    
    # Create DataFrame of final predictions
    final_df = pd.DataFrame(final_predictions)
    
    if len(final_df) > 0:
        # Save post-processed results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(output_dir, f'ensemble_postprocessed_{timestamp}.csv')
        final_df.to_csv(output_path, index=False)
        
        # Save summary
        summary = {
            'timestamp': timestamp,
            'original_predictions': len(pred_df),
            'final_predictions': len(final_df),
            'images_processed': len(image_names),
            'confidence_threshold': confidence_threshold,
            'model_agreement_threshold': model_agreement_threshold,
            'nms_iou_threshold': nms_iou_threshold
        }
        
        summary_path = os.path.join(output_dir, f'postprocessing_summary_{timestamp}.json')
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"Post-processing complete:")
        print(f"  - Original predictions: {len(pred_df)}")
        print(f"  - Final predictions: {len(final_df)}")
        print(f"  - Reduction: {(1 - len(final_df) / len(pred_df)) * 100:.1f}%")
        print(f"Post-processed results saved to: {output_path}")
    else:
        print("No predictions met the threshold criteria after post-processing.")
    
    return final_df

In [29]:
import numpy as np
# Run memory-optimized ensemble predictions
predictions = ensemble_kfold_models_optimized(
    test_images_dir=TEST_IMAGES_DIR,
    cv_results=cv_results,
    conf_threshold=0.3,
    iou_threshold=0.5
)



Creating Memory-Optimized Model Ensemble from K-Fold Models
Found model from fold 1: yolov11_cv_s/fold1_of_5/weights/best.pt
Found model from fold 2: yolov11_cv_s/fold2_of_5/weights/best.pt
Found model from fold 3: yolov11_cv_s/fold3_of_5/weights/best.pt
Found model from fold 4: yolov11_cv_s/fold4_of_5/weights/best.pt
Found model from fold 5: yolov11_cv_s/fold5_of_5/weights/best.pt
Running ensemble inference on: dataset/images/test
Found 1545 images in test directory

Running predictions with model from fold 1

WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs =

In [30]:
all_preds = pd.DataFrame(predictions)
all_preds

,image_name,class,confidence,xmin,ymin,xmax,ymax,model_fold
0,ID_A16nzu.jpg,healthy,0.566845,36.285217,1.956055,950.594543,1273.466553,1
1,ID_A1Euyz.jpg,anthracnose,0.578857,202.237793,96.265869,960.000000,1177.578125,1
2,ID_A1HcV0.jpeg,anthracnose,0.428156,0.000000,119.089760,1483.036377,2576.906738,1
3,ID_A1HcV0.jpeg,anthracnose,0.381752,920.264099,702.710693,2737.285156,3646.960693,1
4,ID_A4ZdJC.jpeg,healthy,0.817828,190.853378,506.534729,2102.094482,4032.000000,1
...,...,...,...,...,...,...,...,...
12275,ID_zyGKPp.jpg,healthy,0.759476,150.393234,103.659218,407.861206,395.058594,5
12276,ID_zyGKPp.jpg,healthy,0.598116,61.712185,163.236160,193.795959,278.837616,5
12277,ID_zyGKPp.jpg,healthy,0.451195,170.110748,266.946594,346.539490,416.000000,5
12278,ID_zyGKPp.jpg,healthy,0.433990,0.000000,42.175117,186.289658,119.743858,5


In [31]:
all_preds.to_csv('ensemble_predictions_full.csv', index=False)

In [32]:

# Optionally perform post-processing
final_predictions = perform_ensemble_post_processing(
    predictions_path='ensemble_predictions_full.csv',
    confidence_threshold=0.5,
    model_agreement_threshold=0.6
)


Performing Ensemble Post-Processing
Post-processing complete:
  - Original predictions: 12280
  - Final predictions: 1866
  - Reduction: 84.8%
Post-processed results saved to: ensemble_postprocessed_20250327_021323.csv


In [33]:
final_predictions

,image_name,class,confidence,xmin,ymin,xmax,ymax,model_agreement,models_detected,total_models
0,ID_A1Euyz.jpg,anthracnose,0.558646,229.457665,143.588281,954.824534,1127.295466,1.0,5,5
1,ID_A4ZdJC.jpeg,healthy,0.797487,247.039530,544.262788,2123.140444,4022.334657,1.0,5,5
2,ID_A5SFUW.jpeg,healthy,0.792499,2.588243,0.967714,2958.226951,3920.471061,1.0,5,5
3,ID_A6Fogi.jpeg,anthracnose,0.644492,98.525042,695.970105,1799.773773,3758.698279,1.0,5,5
4,ID_ABDCyn.jpeg,healthy,0.551010,1092.177790,513.686408,2754.097126,3330.295801,0.8,4,5
...,...,...,...,...,...,...,...,...,...,...
1861,ID_aY2yXb.jpg,anthracnose,0.720392,922.502131,16.979998,3061.476089,4125.783156,0.8,4,5
1862,ID_ewBCM4.jpg,cssvd,0.558495,61.168331,1.512597,725.799264,1079.847279,0.6,3,5
1863,ID_hcfuoO.JPG,anthracnose,0.540795,747.013720,725.660290,1843.741420,2750.976499,0.6,3,5
1864,ID_mHoDGw.JPG,healthy,0.578414,1.084853,1341.693102,1713.892420,3262.127072,0.6,3,5
